# FORGE Worker Notebook — the training executor

This notebook is the thin worker shell of the FORGE AI training platform. The same
logic runs on every provider (Colab primary; Kaggle for long unattended runs;
Hugging Face, Lightning AI and Modal are provider-agnostic slots):

1. **Authenticate** to the Forge API with a worker token (typed or Colab secrets — never stored).
2. **Pull the planned run** and its **pinned dataset snapshot** (fingerprint-verified).
3. **Fine-tune** with Unsloth QLoRA (4-bit, gradient checkpointing, models under 3B).
4. **Stream checkpoints, cost and metrics** to Forge as they happen.
5. **Export weights, fingerprint them (sha256), upload, register** the model version with lineage.
6. **Close the run honestly** — succeeded only if it truly finished.

**Free-tier discipline (law, not preference):** QLoRA 4-bit only, models under 3B
parameters, gradient checkpointing, chunked uploads, resume-from-checkpoint as the
DEFAULT. A killed session restarts exactly where it stopped — local checkpoint and
API checkpoint stay in lockstep.

**What this notebook never does:** it never trains on anything but the pinned
snapshot (the data diet law), never hardcodes a secret, and never promotes a model.
Forge judges (eval suites, safety battery, blind panels), builds the model card,
and promotes experimentally through internal, beta, stable. Only a stable version
serves generation.

## Step 0 — Configure the worker

In [ ]:
# @title Step 0 — Configure (no secrets in code, ever)
import getpass

FORGE_API_URL = "https://YOUR-FORGE-HOST/api"  # @param {type:"string"}
RUN_ID = ""  # @param {type:"string"}
MODEL_ID = "forge-creative-01"  # @param {type:"string"}
OWNER = "sylvester"  # @param {type:"string"}
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  # @param {type:"string"}

# The token is read from Colab secrets when available, typed via getpass
# otherwise. It is never written into this notebook or its outputs.
def _load_token():
    try:
        from google.colab import userdata
        return userdata.get("FORGE_WORKER_TOKEN")
    except Exception:
        return getpass.getpass("FORGE worker token: ")

TOKEN = _load_token()
assert TOKEN, "a FORGE worker token is required"
assert RUN_ID, "RUN_ID is required"

# Free-tier discipline: QLoRA 4-bit only, models at or under 3B parameters,
# gradient checkpointing on, resume-from-checkpoint as the default.
MAX_PARAMS = 3_000_000_000
CHECKPOINT_EVERY = 50   # optimizer steps between streamed checkpoints
LOG_EVERY = 10          # steps between metric/cost posts
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LOCAL_RUN_DIR = "/content/forge-run"
WEIGHTS_OUT_DIR = "/content/forge-weights"
DRIVE_ROOT = "/content/drive/MyDrive/forge"
print("worker configured for run", RUN_ID)

## Step 1 — Pull the planned run and its pinned dataset

The run is a plan created in Forge against a frozen dataset snapshot. The worker
refuses to continue if the run is not executable or the snapshot fingerprint does
not match the fingerprint the run was planned against.

In [ ]:
import json
import requests

def forge(method, path, body=None, ok=(200, 201)):
    r = requests.request(method, FORGE_API_URL + path, json=body,
                         headers={"Authorization": "Bearer " + TOKEN}, timeout=60)
    if r.status_code not in ok:
        raise RuntimeError("Forge refused " + method + " " + path + ": " + str(r.status_code) + " " + r.text[:300])
    return r.json()

run = forge("GET", "/ai-training/runs/" + RUN_ID)
assert run["state"] in ("planned", "running", "paused"), "run-not-executable:" + run["state"]
dataset = forge("GET", "/ai-training/datasets/" + run["datasetId"])
# The snapshot the run was planned against is pinned by fingerprint.
assert dataset["fingerprint"] == run["datasetFingerprint"], "snapshot-fingerprint-mismatch"
plan = run["plan"]
print(json.dumps({"run": run["id"], "state": run["state"], "steps": plan["steps"],
                  "epochs": plan["epochs"], "batchSize": plan["batchSize"],
                  "estimatedCost": plan["estimatedCost"],
                  "datasetFingerprint": run["datasetFingerprint"],
                  "seed": run["seed"]}, indent=2))

## Step 2 — Start (or resume) the run

planned to running starts fresh; paused to running resumes from the last
checkpoint Forge holds (the engine computes the resume point). Resume is the
default, never the exception.

In [ ]:
if run["state"] in ("planned", "paused"):
    run = forge("POST", "/ai-training/runs/" + RUN_ID + "/transition", {"to": "running"})
STEP_COST_UNITS = round(plan["estimatedCost"] / max(plan["steps"], 1), 6)
# The effective batch honors the plan while fitting free-tier memory.
PER_DEVICE_BATCH = min(plan["batchSize"], 4)
GRAD_ACCUM = max(1, plan["batchSize"] // PER_DEVICE_BATCH)
print("run", run["state"], "| step cost", STEP_COST_UNITS,
      "| batch", PER_DEVICE_BATCH, "x", GRAD_ACCUM)

## Step 3 — Mount storage and verify the pinned snapshot

Google Drive now, GCP bucket later — the notebook only needs the snapshot
directory. On Kaggle/Lightning/Modal mount or copy the same snapshot store; the
verification below is identical everywhere.

In [ ]:
import glob
import os

def mount_storage():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass  # Kaggle, Lightning, Modal: mount or copy your snapshot store

mount_storage()
SNAPSHOT_DIR = os.path.join(DRIVE_ROOT, "datasets", run["datasetId"], "v" + str(run["datasetVersion"]))
manifest_path = os.path.join(SNAPSHOT_DIR, "manifest.json")
assert os.path.isfile(manifest_path), "snapshot-missing:" + SNAPSHOT_DIR
with open(manifest_path) as fh:
    manifest = json.load(fh)
assert manifest["fingerprint"] == run["datasetFingerprint"], "snapshot-fingerprint-mismatch"
SNAPSHOT_FILES = sorted(glob.glob(os.path.join(SNAPSHOT_DIR, "*.jsonl")))
assert SNAPSHOT_FILES, "snapshot-empty:" + SNAPSHOT_DIR
print("snapshot verified:", len(SNAPSHOT_FILES), "part(s), fingerprint", run["datasetFingerprint"])

## Step 4 — Load the snapshot into training pairs

The data diet law: only what was fed into the app (with consent) ever trains it.
The intake pipeline exports pairs; anything with an unknown shape is refused
loudly, never silently skipped.

In [ ]:
def to_training_text(record):
    keys = set(record.keys())
    if "instruction" in keys and ("output" in keys or "response" in keys):
        instruction = record["instruction"]
        extra = record.get("input", "")
        output = record.get("output", record.get("response"))
        prompt = instruction + (chr(10) + extra if extra else "")
    elif "prompt" in keys and "response" in keys:
        prompt, output = record["prompt"], record["response"]
    elif "text" in keys:
        return record["text"]
    else:
        raise ValueError("unknown-record-shape:" + ",".join(sorted(keys)))
    return "### Input:" + chr(10) + prompt + chr(10) + chr(10) + "### Response:" + chr(10) + output

pairs = []
for path in SNAPSHOT_FILES:
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                pairs.append({"text": to_training_text(json.loads(line))})
assert pairs, "snapshot-had-no-records"
print(len(pairs), "training pairs loaded")

## Step 5 — Fine-tune (Unsloth QLoRA) and stream progress to Forge

Every logging step posts a metric point and its cost; every CHECKPOINT_EVERY steps
posts a checkpoint. Forge refuses anything outside the plan by name — the budget
is enforced server-side, never guessed here.

In [ ]:
from datasets import Dataset
from transformers import TrainerCallback, TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
import os

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True)
n_params = sum(p.numel() for p in model.parameters())
assert n_params <= MAX_PARAMS, "model-over-free-tier-limit:" + str(n_params)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=run["seed"])

class ForgeReporter(TrainerCallback):
    """Streams loss, cost and checkpoints to Forge; nothing is silent."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        step = int(state.global_step)
        loss = float(logs["loss"])
        forge("POST", "/ai-training/runs/" + RUN_ID + "/metrics", {"step": step, "loss": loss})
        forge("POST", "/ai-training/runs/" + RUN_ID + "/cost",
              {"step": step, "units": round(STEP_COST_UNITS * LOG_EVERY, 6)})
        if step % CHECKPOINT_EVERY == 0:
            forge("POST", "/ai-training/runs/" + RUN_ID + "/checkpoint", {"step": step, "loss": loss})
        print("step", step, "loss", round(loss, 4), flush=True)

os.makedirs(LOCAL_RUN_DIR, exist_ok=True)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=Dataset.from_list(pairs),
    dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=plan["epochs"],
        learning_rate=2e-4, lr_scheduler_type="linear", warmup_ratio=0.03,
        logging_steps=LOG_EVERY, save_steps=CHECKPOINT_EVERY, save_total_limit=2,
        output_dir=LOCAL_RUN_DIR, optim="adamw_8bit", seed=run["seed"],
        report_to="none", save_safetensors=True),
    callbacks=[ForgeReporter()])

# resume_from_checkpoint is the default: a killed session restarts exactly
# where it stopped, locally and on the Forge side.
HAS_LOCAL_CHECKPOINT = any(f.startswith("checkpoint-") for f in os.listdir(LOCAL_RUN_DIR))
trainer_stats = trainer.train(resume_from_checkpoint=LOCAL_RUN_DIR if HAS_LOCAL_CHECKPOINT else None)
print("training finished:", trainer_stats.metrics)

## Step 6 — Export weights, fingerprint, upload, register with lineage

The weights fingerprint is sha256 over every exported weight file (sorted by
name, size, content) — deterministic and idempotent. Registration carries the
lineage: which pinned dataset and which run produced the build. Re-running this
cell after a kill reuses the version Forge already registered for the same
fingerprint.

In [ ]:
import hashlib
import os

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

os.makedirs(WEIGHTS_OUT_DIR, exist_ok=True)
model.save_pretrained_merged(WEIGHTS_OUT_DIR, tokenizer, save_method="merged_16bit")
names = sorted(f for f in os.listdir(WEIGHTS_OUT_DIR)
               if f.endswith((".safetensors", ".json")))
digest = hashlib.sha256()
for name in names:
    full = os.path.join(WEIGHTS_OUT_DIR, name)
    digest.update(name.encode())
    digest.update(str(os.path.getsize(full)).encode())
    digest.update(sha256_file(full).encode())
WEIGHTS_FINGERPRINT = "wf_sha256_" + digest.hexdigest()[:32]
print("weights fingerprint:", WEIGHTS_FINGERPRINT)

def chunked_upload(src_dir, dst_dir, chunk_mb=8):
    """Chunked upload with resume: files that already match are skipped."""
    os.makedirs(dst_dir, exist_ok=True)
    for name in sorted(os.listdir(src_dir)):
        src = os.path.join(src_dir, name)
        dst = os.path.join(dst_dir, name)
        if os.path.isfile(src) and os.path.isfile(dst):
            if os.path.getsize(dst) == os.path.getsize(src) and sha256_file(dst) == sha256_file(src):
                continue
        with open(src, "rb") as fin, open(dst, "wb") as fout:
            while True:
                block = fin.read(chunk_mb << 20)
                if not block:
                    break
                fout.write(block)
        print("uploaded", name)

chunked_upload(WEIGHTS_OUT_DIR, os.path.join(DRIVE_ROOT, "weights", run["runFingerprint"]))

registration = forge("POST", "/ai-training/models/register", {
    "modelId": MODEL_ID,
    "owners": [OWNER],
    "weightsFingerprint": WEIGHTS_FINGERPRINT,
    "lineage": {"datasetId": run["datasetId"], "runFingerprint": run["runFingerprint"]},
})
VERSION = registration["version"]
print("registered", MODEL_ID, "version", VERSION,
      "(reused existing build)" if registration.get("reused") else "(new build)")

## Step 7 — Close the run honestly

succeeded only because training finished AND the weights were registered. If a
cell above failed, close the run as failed instead (from the Forge app or by
posting {"to": "failed"}) — a run is never faked green.

In [ ]:
forge("POST", "/ai-training/runs/" + RUN_ID + "/transition", {"to": "succeeded"})
print("run succeeded:", RUN_ID)

## Step 8 — What Forge does next (the worker never promotes)

Forge judges the candidate: eval suites, safety battery, blind panels — then a
model card — then staged promotion: experimental to internal to beta to stable.
Only a stable version serves generation. The worker watches; it never promotes
and never skips a gate.

In [ ]:
import time

for _ in range(60):
    versions = forge("GET", "/ai-training/models/" + MODEL_ID)
    latest = max(versions, key=lambda v: v.get("version", 0))
    print("version", latest.get("version"), "stage", latest.get("stage"))
    if latest.get("stage") in ("beta", "stable"):
        break
    time.sleep(30)
print("Promotion path: experimental -> internal -> beta -> stable.")
print("Gates judge. The worker waits. Serving starts only after stable.")